# Validación del Dataset

**Objetivo:** Verificar la calidad, distribución y consistencia del dataset generado
**Duración estimada:** 30 minutos

---

## Contenido

1. [Setup](#setup)
2. [Carga del Dataset](#carga-del-dataset)
3. [Validación Estructural](#validacion-estructural)
4. [Análisis de Distribución](#analisis-distribucion)
5. [Detección de Duplicados](#deteccion-duplicados)
6. [Validación de Etiquetas](#validacion-etiquetas)
7. [Pruebas de Parseabilidad](#pruebas-de-parseabilidad)
8. [Reporte Final de Calidad](#reporte-final)

---

## 1. Setup

In [ ]:
import sys
import json
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, '../..')

from dataset_generator import (
    SyntheticDataset,
    DatasetExporter,
    DatasetSplit
)
from app.core.parser import parse_pseudocode

import matplotlib.pyplot as plt
import numpy as np

print("Setup completado")

---

## 2. Carga del Dataset

In [ ]:
# Intentar cargar dataset previamente generado (JSON) y buscar .txt en fixtures/data
json_paths = [
    Path("../../data/datasets/synthetic_algorithms.json"),
    Path("../../data/datasets/labeled_algorithms.json"),
]

txt_search_paths = [
    Path("../../tests/fixtures/sample_algorithms"),
    Path("../../data/algorithms"),
]

datasets_cargados = []

# Cargar JSONs si existen
for path in json_paths:
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            datos = json.load(f)
            datasets_cargados.append({"archivo": path.name, "datos": datos})
        print(f"Cargado JSON: {path.name} ({len(datos.get('ejemplos', []))} ejemplos)")
    else:
        print(f"No encontrado JSON: {path.name}")

# Buscar archivos .txt en fixtures y data/algorithms
txt_files = []
for base in txt_search_paths:
    if base.exists():
        for p in base.rglob("*.txt"):
            txt_files.append(p)
    else:
        print(f"No encontrado path de textos: {base}")

if txt_files:
    ejemplos_txt = []
    for p in sorted(txt_files):
        try:
            with open(p, "r", encoding="utf-8") as f:
                codigo = f.read().strip()
            ejemplo = {
                "id": p.stem,
                "big_o": None,
                "patron": None,
                "codigo": codigo,
                "fuente": str(p)
            }
            ejemplos_txt.append(ejemplo)
        except Exception as e:
            print(f"Error leyendo {p}: {e}")
    datasets_cargados.append({"archivo": "txt_files", "datos": {"ejemplos": ejemplos_txt}})
    print(f"Cargados {len(ejemplos_txt)} ejemplos desde archivos .txt")
else:
    print("No se encontraron archivos .txt en fixtures o data/algorithms")

# Si no hay datasets previos, crear dataset mínimo para demostración
if not datasets_cargados:
    print("Creando dataset mínimo de demostración...")
    ALGORITMOS_DEMO = [
        {"id": "d001", "big_o": "O(n)", "patron": "BRUTE_FORCE", "codigo": "algorithm sumar(A[], n)\nbegin\n    total <- 0\n    for i <- 1 to n do\n        total <- total + A[i]\n    end\n    return total\nend"},
        {"id": "d002", "big_o": "O(n^2)", "patron": "SORTING", "codigo": "algorithm burbuja(A[], n)\nbegin\n    for i <- 1 to n do\n        for j <- 1 to n - i do\n            if (A[j] > A[j+1]) then\n                temp <- A[j]\n                A[j] <- A[j+1]\n                A[j+1] <- temp\n            end\n        end\n    end\nend"},
        {"id": "d003", "big_o": "O(log n)", "patron": "SEARCHING", "codigo": "algorithm binSearch(A[], lo, hi, k)\nbegin\n    while (lo <= hi) do\n        mid <- floor((lo + hi) / 2)\n        if (A[mid] = k) then\n            return mid\n        end\n        if (A[mid] < k) then\n            lo <- mid + 1\n        else\n            hi <- mid - 1\n        end\n    end\n    return -1\nend"},
        {"id": "d004", "big_o": "O(2^n)", "patron": "RECURSIVE", "codigo": "algorithm fib(n)\nbegin\n    if (n <= 1) then\n        return n\n    end\n    return fib(n-1) + fib(n-2)\nend"},
        {"id": "d005", "big_o": "O(n log n)", "patron": "DIVIDE_AND_CONQUER", "codigo": "algorithm mergeSort(A[], p, r)\nbegin\n    if (p < r) then\n        q <- floor((p + r) / 2)\n        call mergeSort(A, p, q)\n        call mergeSort(A, q+1, r)\n    end\nend"},
    ]
    datasets_cargados = [{"archivo": "demo", "datos": {"ejemplos": ALGORITMOS_DEMO}}]

# Combinar todos los datasets
todos_los_ejemplos = []
for ds in datasets_cargados:
    todos_los_ejemplos.extend(ds["datos"].get("ejemplos", []))

print(f"\nTotal de ejemplos en el dataset combinado: {len(todos_los_ejemplos)}")

---

## 3. Validación Estructural

In [ ]:
CAMPOS_REQUERIDOS = ["id", "big_o", "patron", "codigo"]
CAMPOS_OPCIONALES = ["omega", "espacio", "estructuras", "confianza", "fuente"]

def validar_estructura(ejemplo):
    """Verifica que un ejemplo tenga todos los campos requeridos."""
    errores = []
    for campo in CAMPOS_REQUERIDOS:
        if campo not in ejemplo or ejemplo[campo] is None:
            errores.append(f"Campo requerido faltante: '{campo}'")
    return errores

print("VALIDACIÓN ESTRUCTURAL:")
ejemplos_validos = []
ejemplos_invalidos = []
campos_faltantes_total = Counter()

for ejemplo in todos_los_ejemplos:
    errores = validar_estructura(ejemplo)
    if errores:
        ejemplos_invalidos.append({"ejemplo": ejemplo, "errores": errores})
        for error in errores:
            campo = error.split("'")[1] if "'" in error else "desconocido"
            campos_faltantes_total[campo] += 1
    else:
        ejemplos_validos.append(ejemplo)

print(f"Ejemplos válidos:   {len(ejemplos_validos)}/{len(todos_los_ejemplos)}")
print(f"Ejemplos inválidos: {len(ejemplos_invalidos)}/{len(todos_los_ejemplos)}")

if campos_faltantes_total:
    print("\nCampos faltantes más frecuentes:")
    for campo, count in campos_faltantes_total.most_common():
        print(f"  '{campo}': {count} veces")
else:
    print("Todos los campos requeridos están presentes.")

---

## 4. Análisis de Distribución

In [ ]:
# Distribución por Big O
big_o_dist = Counter(e.get("big_o") for e in ejemplos_validos)

# Distribución por patrón
patron_dist = Counter(e.get("patron") for e in ejemplos_validos)

print("ANÁLISIS DE DISTRIBUCIÓN DEL DATASET:")

print("\nPor Complejidad Temporal (Big O):")
total = len(ejemplos_validos)
for big_o, count in sorted(big_o_dist.items(), key=lambda x: -x[1]):
    porcentaje = count / total * 100
    barra = "#" * int(porcentaje / 2)
    print(f"  {str(big_o):<15}: {barra:<25} {count:>4} ({porcentaje:.1f}%)")

print("\nPor Patrón Algorítmico:")
for patron, count in sorted(patron_dist.items(), key=lambda x: -x[1]):
    porcentaje = count / total * 100
    barra = "#" * int(porcentaje / 2)
    print(f"  {str(patron):<25}: {barra:<25} {count:>4} ({porcentaje:.1f}%)")

# Verificar balance del dataset
max_count = max(big_o_dist.values()) if big_o_dist else 1
min_count = min(big_o_dist.values()) if big_o_dist else 0
ratio_desbalance = max_count / max(min_count, 1)
print(f"\nRatio de desbalance (Big O): {ratio_desbalance:.1f}x")
if ratio_desbalance > 3:
    print("ADVERTENCIA: Dataset desbalanceado. Considerar oversampling/undersampling.")
else:
    print("Dataset suficientemente balanceado para uso general.")

---

## 5. Detección de Duplicados

In [ ]:
def calcular_hash_codigo(codigo):
    """Calcula hash del código normalizando espacios."""
    codigo_normalizado = " ".join(codigo.strip().split())
    return hashlib.md5(codigo_normalizado.encode()).hexdigest()

print("DETECCIÓN DE DUPLICADOS:")
hashes_vistos = {}
duplicados = []
duplicados_cercanos = []

for ejemplo in ejemplos_validos:
    codigo = ejemplo.get("codigo", "")
    hash_codigo = calcular_hash_codigo(codigo)
    
    if hash_codigo in hashes_vistos:
        duplicados.append({
            "id_actual": ejemplo.get("id"),
            "id_original": hashes_vistos[hash_codigo],
            "tipo": "exacto"
        })
    else:
        hashes_vistos[hash_codigo] = ejemplo.get("id")

# Detección de duplicados por nombre de algoritmo
nombres_vistos = defaultdict(list)
for ejemplo in ejemplos_validos:
    codigo = ejemplo.get("codigo", "")
    try:
        ast = parse_pseudocode(codigo)
        nombre = ast.algorithm.name
        nombres_vistos[nombre].append(ejemplo.get("id"))
    except Exception:
        pass

for nombre, ids in nombres_vistos.items():
    if len(ids) > 1:
        duplicados_cercanos.append({"nombre": nombre, "ids": ids})

print(f"Duplicados exactos encontrados:    {len(duplicados)}")
print(f"Algoritmos con mismo nombre:       {len(duplicados_cercanos)}")
print(f"Ejemplos únicos:                   {len(ejemplos_validos) - len(duplicados)}")

if duplicados_cercanos:
    print("\nAlgoritmos con nombre repetido:")
    for dup in duplicados_cercanos[:5]:
        print(f"  '{dup['nombre']}': {dup['ids']}")

---

## 6. Validación de Etiquetas

In [ ]:
# Verificar consistencia de las etiquetas
NOTACIONES_VALIDAS_BIG_O = {
    "O(1)", "O(log n)", "O(n)", "O(n log n)",
    "O(n^2)", "O(n^3)", "O(2^n)", "O(n!)"
}

PATRONES_VALIDOS = {
    "BRUTE_FORCE", "RECURSIVE", "DIVIDE_AND_CONQUER",
    "DYNAMIC_PROGRAMMING", "GREEDY", "BACKTRACKING",
    "BRANCH_AND_BOUND", "SORTING", "SEARCHING",
    "QUANTUM", "BIO_INSPIRED", "APPROXIMATION"
}

print("VALIDACIÓN DE ETIQUETAS:")
big_o_invalidos = []
patrones_invalidos = []

for ejemplo in ejemplos_validos:
    big_o = str(ejemplo.get("big_o", ""))
    patron = str(ejemplo.get("patron", "")).replace("PatternType.", "")
    
    if big_o not in NOTACIONES_VALIDAS_BIG_O:
        big_o_invalidos.append({"id": ejemplo.get("id"), "valor": big_o})
    
    if patron not in PATRONES_VALIDOS and patron not in ("None", ""):
        patrones_invalidos.append({"id": ejemplo.get("id"), "valor": patron})

print(f"Big O con notación no estándar: {len(big_o_invalidos)}")
if big_o_invalidos:
    for item in big_o_invalidos[:5]:
        print(f"  [{item['id']}]: '{item['valor']}'")

print(f"Patrones no reconocidos:        {len(patrones_invalidos)}")
if patrones_invalidos:
    for item in patrones_invalidos[:5]:
        print(f"  [{item['id']}]: '{item['valor']}'")

---

## 7. Pruebas de Parseabilidad

In [ ]:
print("PRUEBAS DE PARSEABILIDAD:")
parseables = 0
no_parseables = []

for ejemplo in ejemplos_validos:
    codigo = ejemplo.get("codigo", "")
    try:
        ast = parse_pseudocode(codigo)
        parseables += 1
    except Exception as e:
        no_parseables.append({
            "id": ejemplo.get("id"),
            "error": str(e)[:60]
        })

print(f"Algoritmos parseables:     {parseables}/{len(ejemplos_validos)}")
print(f"Algoritmos no parseables:  {len(no_parseables)}/{len(ejemplos_validos)}")

if no_parseables:
    print("\nEjemplos con error de parsing:")
    for item in no_parseables[:5]:
        print(f"  [{item['id']}]: {item['error']}")

---

## 8. Reporte Final de Calidad

In [ ]:
# Calcular score global de calidad
score_estructura = len(ejemplos_validos) / max(len(todos_los_ejemplos), 1)
score_duplicados = 1 - (len(duplicados) / max(len(ejemplos_validos), 1))
score_parseabilidad = parseables / max(len(ejemplos_validos), 1)
score_etiquetas = 1 - ((len(big_o_invalidos) + len(patrones_invalidos)) / max(len(ejemplos_validos), 1))
score_balance = min(1.0, 3.0 / ratio_desbalance) if ratio_desbalance > 0 else 1.0

score_global = (
    score_estructura * 0.25 +
    score_duplicados * 0.15 +
    score_parseabilidad * 0.30 +
    score_etiquetas * 0.20 +
    score_balance * 0.10
)

print("REPORTE FINAL DE CALIDAD DEL DATASET")
print(f"Integridad estructural:    {score_estructura:.1%}")
print(f"Unicidad (sin duplicados): {score_duplicados:.1%}")
print(f"Parseabilidad:             {score_parseabilidad:.1%}")
print(f"Calidad de etiquetas:      {score_etiquetas:.1%}")
print(f"Balance de clases:         {score_balance:.1%}")
print(f"SCORE GLOBAL DE CALIDAD:   {score_global:.1%}")

if score_global >= 0.9:
    print("ESTADO: EXCELENTE - El dataset está listo para uso en producción")
elif score_global >= 0.7:
    print("ESTADO: BUENO - El dataset es usable con precauciones menores")
elif score_global >= 0.5:
    print("ESTADO: REGULAR - Requiere mejoras antes de uso en producción")
else:
    print("ESTADO: DEFICIENTE - Requiere revisión significativa")

# Exportar reporte
reporte = {
    "score_global": score_global,
    "total_ejemplos": len(todos_los_ejemplos),
    "ejemplos_validos": len(ejemplos_validos),
    "duplicados": len(duplicados),
    "no_parseables": len(no_parseables),
    "distribucion_big_o": dict(big_o_dist),
    "distribucion_patron": dict(patron_dist)
}
output_report = Path("../../data/datasets/validation_report.json")
output_report.parent.mkdir(parents=True, exist_ok=True)
with open(output_report, "w") as f:
    json.dump(reporte, f, indent=2, ensure_ascii=False)
print(f"\nReporte guardado en: {output_report}")

---

## Proximos Pasos

- **labeling_process.ipynb**: Revisar y corregir etiquetas problemáticas detectadas
- **synthetic_algorithms.ipynb**: Generar más datos para clases sub-representadas